### ⚠️ Se você já rodou este notebook antes (erro 429 / "xet-read-token")

Esse dataset usa o backend de armazenamento **Xet** da Hugging Face, que tem um limite de requisições próprio (independente do rate limit normal) — é isso que causa o erro `429 Too Many Requests` no `xet-read-token`, mesmo autenticado.

A correção é desativar o Xet e cair no download HTTP clássico (mais lento por arquivo, mas sem esse limite). Isso só funciona se for feito **antes de qualquer import da `huggingface_hub`** no kernel.

**Se este kernel já rodou alguma célula antes:** vá em **Run → Restart Kernel** (ou o ícone de reiniciar no topo) antes de rodar as células a partir daqui — senão o ajuste abaixo não tem efeito.

In [1]:
import os

# Precisa ser definido ANTES de qualquer "import huggingface_hub" no kernel
# (o huggingface_hub lê essa variável só uma vez, na primeira importação).
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("Xet desativado — downloads vão usar o caminho HTTP clássico.")

Xet desativado — downloads vão usar o caminho HTTP clássico.


# Treino YOLOv8 — STRIDE Architecture Components

Notebook para treinar um detector de componentes de arquitetura cloud (32 classes) usando o dataset [`guillherms/stride-architecture-components-v1`](https://huggingface.co/datasets/guillherms/stride-architecture-components-v1) (4190 imagens, formato YOLO, já dividido em train/val/test).

**Antes de rodar, nas configurações do notebook (ícone de engrenagem à direita):**
1. Ligue **Internet** (exige verificação de telefone na conta Kaggle, se ainda não fez).
2. Escolha um **Accelerator** com GPU: `GPU P100` (1 GPU, mais simples) ou `GPU T4 x2` (este notebook usa só a GPU 0, então tanto faz).
3. (Opcional, recomendado) Anexe o dataset sintético de balanceamento de classes — veja a seção 1.5 abaixo.
4. Depois de rodar as células uma vez para conferir que está tudo certo, clique em **Save Version → Save & Run All (Commit)** — isso deixa o notebook rodando em background, dá pra fechar a aba e voltar depois (até ~9-12h de sessão).
5. Ao terminar, baixe os arquivos na aba **Output** da versão salva: `best.pt` (pesos) e `stride_yolov8s_run.zip` (métricas e gráficos).

**Expectativa de tempo:** treino de YOLOv8s em ~3000 imagens de treino, GPU T4/P100, 100 épocas com early stopping (patience=20) — tipicamente 1 a 2 horas. Bem dentro das ~30h/semana grátis do Kaggle.

In [2]:
!pip install -q -U ultralytics huggingface_hub pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 98.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


## 1. Baixar o dataset direto do Hugging Face Hub

Não precisa subir nada do seu computador — o dataset já está público no HF Hub.

**Autenticação no Hugging Face (evita o rate limit de download sem login):**

1. No menu do notebook, vá em **Add-ons → Secrets**.
2. Clique em **Add a new secret**: Label = `HF_TOKEN`, Value = cole seu token (o mesmo que você já gerou em huggingface.co/settings/tokens, tipo Read-Only).
3. Deixe o toggle **Attached** ligado para este notebook.
4. Feche o painel de Secrets e rode a célula abaixo.

Guardar o token assim (em vez de escrito direto no código) evita que ele vaze se este notebook for commitado no seu repositório do GitHub.

In [3]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN não encontrado nos Secrets deste notebook.\n"
        "Vá em Add-ons -> Secrets, adicione um secret com Label 'HF_TOKEN' "
        "contendo seu token do huggingface.co/settings/tokens, e deixe 'Attached' ligado."
    )

login(token=hf_token)

!pip install -q hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.8 MB/s eta 0:00:00


In [4]:
from huggingface_hub import snapshot_download

DATASET_DIR = snapshot_download(
    repo_id="guillherms/stride-architecture-components-v1",
    repo_type="dataset",
    local_dir="/kaggle/working/stride-dataset",
    max_workers=4,  # menos requisições em paralelo, mais seguro contra 429
)
print("Dataset baixado em:", DATASET_DIR)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6195 files:   0%|          | 0/6195 [00:00<?, ?it/s]

HTTP Error 429 thrown while requesting GET https://huggingface.co/datasets/guillherms/stride-architecture-components-v1/resolve/0612f31161fd79ba887a4a77b0dba20f0d05e6b9/train/images/fe772466-aws_solution_20260202_43_degrade80.jpg
Rate limited. Waiting 172.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/guillherms/stride-architecture-components-v1/resolve/0612f31161fd79ba887a4a77b0dba20f0d05e6b9/train/images/fe772466-aws_solution_20260202_43_degrade80.npy
Rate limited. Waiting 172.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/guillherms/stride-architecture-components-v1/resolve/0612f31161fd79ba887a4a77b0dba20f0d05e6b9/train/images/fe772466-aws_solution_20260202_43_gamma_hi.jpg
Rate limited. Waiting 172.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/guillherms/stride-architecture-components-v1/resolve/0612f31161fd79b

Dataset baixado em: /kaggle/working/stride-dataset


In [5]:
import yaml
from pathlib import Path

DATA_DIR = Path("/kaggle/working/stride-dataset")

with open(DATA_DIR / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)

# Reescreve o path como absoluto para não depender do cwd de onde o YOLO é chamado
data_cfg["path"] = str(DATA_DIR)

data_kaggle_yaml = DATA_DIR / "data_kaggle.yaml"
with open(data_kaggle_yaml, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

print(data_cfg)

for split in ["train", "val", "test"]:
    n_images = len(list((DATA_DIR / split / "images").glob("*")))
    n_labels = len(list((DATA_DIR / split / "labels").glob("*.txt")))
    print(f"{split}: {n_images} imagens, {n_labels} labels")

{'path': '/kaggle/working/stride-dataset', 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'names': {0: 'actor_user', 1: 'actor_admin', 2: 'edge_ddos_protection', 3: 'edge_cdn', 4: 'edge_waf', 5: 'edge_gateway', 6: 'edge_portal', 7: 'external_entry_point', 8: 'integration_orchestrator', 9: 'integration_messaging', 10: 'compute_load_balancer', 11: 'compute_service', 12: 'compute_worker', 13: 'data_database', 14: 'data_cache', 15: 'data_storage', 16: 'security_identity_provider', 17: 'security_key_management', 18: 'obs_monitoring', 19: 'obs_audit', 20: 'external_backend_service', 21: 'external_saas_service', 22: 'external_web_service', 23: 'communication_service', 24: 'backup_service', 25: 'boundary_cloud', 26: 'boundary_region', 27: 'boundary_resource_group', 28: 'boundary_vpc_or_vnet', 29: 'boundary_subnet_public', 30: 'boundary_subnet_private', 31: 'boundary_autoscaling_group'}}
train: 2980 imagens, 1490 labels
val: 840 imagens, 420 labels
test: 230 imagens, 230 l

## 1.5 (Opcional) Mesclar dataset sintético de balanceamento de classes

O dataset real tem duas classes com **zero exemplos** (`actor_admin` e `integration_messaging`) e várias outras sem nenhum exemplo no split de validação — sem isso, o modelo nunca vai aprender essas classes, não importa quantas épocas rode.

Para corrigir, gerei um lote de 300 diagramas sintéticos reforçando essas classes, dividido em 3 arquivos (por causa do limite de tamanho de upload): `stride_synthetic_part1.zip`, `stride_synthetic_part2.zip`, `stride_synthetic_part3.zip`. Antes de rodar a célula abaixo:

1. Crie um novo dataset no Kaggle (kaggle.com → **Datasets → New Dataset**) e arraste os **3 arquivos zip juntos** na mesma criação.
2. Neste notebook, clique em **+ Add Input** (painel direito) e anexe o dataset que você acabou de criar.
3. Rode a célula abaixo — ela procura automaticamente por pastas/zips `train/images` e `val/images` em `/kaggle/input/` (extraindo zips soltos se precisar) e mescla tudo no dataset real.

Se você pular esse passo, o notebook treina normalmente só com o dataset real (sem o reforço de classes).

In [6]:
import shutil, glob, zipfile

def find_synthetic_splits():
    # Extrai qualquer .zip solto em /kaggle/input (caso o Kaggle não tenha
    # auto-extraído os 3 arquivos que você anexou) para uma pasta de trabalho.
    extract_dir = Path("/kaggle/working/synthetic_extracted")
    extract_dir.mkdir(exist_ok=True)
    for zip_path in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

    search_roots = ["/kaggle/input", str(extract_dir)]
    train_dirs, val_dirs = [], []
    for root in search_roots:
        train_dirs += glob.glob(f"{root}/**/train/images", recursive=True)
        val_dirs += glob.glob(f"{root}/**/val/images", recursive=True)
    return train_dirs, val_dirs


train_dirs, val_dirs = find_synthetic_splits()

if train_dirs or val_dirs:
    print(f"Encontrados {len(train_dirs)} diretório(s) train e {len(val_dirs)} val sintéticos.")
    for split, src_dirs in [("train", train_dirs), ("val", val_dirs)]:
        dst_images = DATA_DIR / split / "images"
        dst_labels = DATA_DIR / split / "labels"
        n_copied = 0
        for images_dir in src_dirs:
            images_dir = Path(images_dir)
            labels_dir = images_dir.parent / "labels"
            for img_path in images_dir.glob("*"):
                label_path = labels_dir / f"{img_path.stem}.txt"
                if not label_path.exists():
                    continue
                shutil.copy(img_path, dst_images / f"synth_{img_path.name}")
                shutil.copy(label_path, dst_labels / f"synth_{label_path.name}")
                n_copied += 1
        print(f"{split}: +{n_copied} imagens sintéticas mescladas")

    print("\nContagem final por split:")
    for split in ["train", "val", "test"]:
        n_images = len(list((DATA_DIR / split / "images").glob("*")))
        print(f"  {split}: {n_images} imagens")
else:
    print(
        "Nenhum dataset sintético encontrado em /kaggle/input — treinando só com o dataset real.\n"
        "Para reforçar classes raras/ausentes (actor_admin, integration_messaging etc.), crie um "
        "Kaggle Dataset com os 3 arquivos stride_synthetic_part*.zip e anexe aqui via '+ Add Input' "
        "antes de rodar esta célula de novo."
    )

Encontrados 5 diretório(s) train e 2 val sintéticos.
train: +516 imagens sintéticas mescladas
val: +130 imagens sintéticas mescladas

Contagem final por split:
  train: 3496 imagens
  val: 970 imagens
  test: 230 imagens


## 2. Treinar o YOLOv8

Usando `yolov8s.pt` (small) como ponto de partida — melhor precisão que o `nano` com custo de tempo ainda tranquilo para ~4000 imagens numa GPU T4/P100. Troque para `"yolov8n.pt"` na linha abaixo se quiser priorizar velocidade.

In [7]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data=str(data_kaggle_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,       # early stopping se não melhorar em 20 épocas
    device=0,
    cache=True,        # acelera épocas seguintes cacheando imagens em RAM/disco
    plots=True,        # gera matriz de confusão e curvas PR — ótimo material pro vídeo
    project="/kaggle/working/runs",
    name="stride_yolov8s",
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/stride-dataset/data_kaggle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, for

## 3. Avaliar no split de validação e no de teste

In [8]:
metrics = model.val(data=str(data_kaggle_yaml), split="val")
print("[VAL] mAP50-95:", metrics.box.map)
print("[VAL] mAP50:", metrics.box.map50)
print("[VAL] Precision média:", metrics.box.mp)
print("[VAL] Recall médio:", metrics.box.mr)

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,137,968 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1236.0±556.3 MB/s, size: 61.6 KB)
val: Scanning /kaggle/working/stride-dataset/val/labels.cache... 550 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 288.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 4.7it/s 7.4s
                   all        550       2969      0.542      0.891      0.723      0.595
            actor_user        149        219      0.738      0.945      0.923      0.601
           actor_admin         13         13      0.258      0.966      0.556      0.514
  edge_ddos_protection         52         52      0.619      0.962      0.896      0.555
              edge_cdn        101        111      0.703      0.811      0.824      0.664
              edge_waf        11

In [9]:
test_metrics = model.val(data=str(data_kaggle_yaml), split="test")
print("[TEST] mAP50-95:", test_metrics.box.map)
print("[TEST] mAP50:", test_metrics.box.map50)

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1732.0±970.9 MB/s, size: 79.1 KB)
val: Scanning /kaggle/working/stride-dataset/test/labels... 230 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 230/230 1.2Kit/s 0.2s
val: /kaggle/working/stride-dataset/test/images/6b0512ca-aws_solution_20260202_58.jpg: 1 duplicate labels removed
val: /kaggle/working/stride-dataset/test/images/6b0512ca-aws_solution_20260202_58_blur1.jpg: 1 duplicate labels removed
val: /kaggle/working/stride-dataset/test/images/6b0512ca-aws_solution_20260202_58_bw.jpg: 1 duplicate labels removed
val: /kaggle/working/stride-dataset/test/images/6b0512ca-aws_solution_20260202_58_contrast.jpg: 1 duplicate labels removed
val: /kaggle/working/stride-dataset/test/images/6b0512ca-aws_solution_20260202_58_degrade80.jpg: 1 duplicate labels removed
val: /kaggle/working/stride-dataset/test/images/6b0512ca-aws_solution_20260202_58_gamma_hi.jpg: 

## 4. Conferência visual rápida em algumas imagens de teste

In [10]:
import glob

sample_images = glob.glob(str(DATA_DIR / "test" / "images" / "*"))[:6]
model.predict(
    source=sample_images,
    imgsz=640,
    conf=0.25,
    save=True,
    project="/kaggle/working/runs",
    name="stride_yolov8s_preds",
)
print("Predições salvas em /kaggle/working/runs/stride_yolov8s_preds")


0: 640x640 (no detections), 13.3ms
1: 640x640 1 actor_user, 1 edge_gateway, 1 integration_orchestrator, 13.3ms
2: 640x640 1 backup_service, 1 boundary_cloud, 1 boundary_vpc_or_vnet, 1 boundary_subnet_private, 13.3ms
3: 640x640 2 compute_services, 2 boundary_regions, 2 boundary_vpc_or_vnets, 13.3ms
4: 640x640 1 backup_service, 1 boundary_cloud, 1 boundary_vpc_or_vnet, 1 boundary_subnet_private, 13.3ms
5: 640x640 1 edge_gateway, 1 security_identity_provider, 13.3ms
Speed: 2.5ms preprocess, 13.3ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /kaggle/working/runs/stride_yolov8s_preds
Predições salvas em /kaggle/working/runs/stride_yolov8s_preds


## 5. Empacotar pesos e resultados para download

In [11]:
import shutil

best_weights = Path("/kaggle/working/runs/stride_yolov8s/weights/best.pt")
shutil.copy(best_weights, "/kaggle/working/best.pt")

shutil.make_archive("/kaggle/working/stride_yolov8s_run", "zip", "/kaggle/working/runs/stride_yolov8s")

print("Pronto! Depois do 'Save & Run All', baixe na aba Output:")
print(" - best.pt                 (pesos do modelo treinado)")
print(" - stride_yolov8s_run.zip  (métricas, gráficos, matriz de confusão)")

Pronto! Depois do 'Save & Run All', baixe na aba Output:
 - best.pt                 (pesos do modelo treinado)
 - stride_yolov8s_run.zip  (métricas, gráficos, matriz de confusão)


## Próximos passos

Depois de baixar `best.pt`:
1. Coloque o arquivo em `hackathon-stride-ai/slm/weights/stride_yolov8s/best.pt` no seu repositório local.
2. Ele será carregado por `slm/evaluate.py` e pela API (`api/main.py`) para gerar a lista de componentes que alimenta os validadores Claude/OpenAI e o Consensus Engine.
3. Guarde as métricas de mAP (val e test) e a matriz de confusão do `.zip` — são ótimo material para a seção de métricas do vídeo final.